# Lane Segmentation on TuSimple with U-Net

Colab pipeline: установка библиотек, подключение Google Drive, загрузка TuSimple, генерация масок, обучение U-Net, Dice/IoU метрики и визуализация результата.

## 1. Установка библиотек

In [ ]:
!pip install tensorflow opencv-python matplotlib tqdm

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Импорт библиотек

In [ ]:
import json
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers
from tqdm import tqdm

## 4. Пути и настройки

Поменяй `DATASET_PATH` под свой Google Drive.

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/TUSimple/train_set"

IMG_HEIGHT = 256
IMG_WIDTH = 256
LIMIT = 200

JSON_FILES = [
    os.path.join(DATASET_PATH, "label_data_0313.json"),
    # os.path.join(DATASET_PATH, "label_data_0531.json"),
    # os.path.join(DATASET_PATH, "label_data_0601.json"),
]

## 5. Конвертация TuSimple JSON в маски

В TuSimple `lanes` хранит x-координаты, а y-координаты лежат в `h_samples`. Поэтому их нужно объединять через `zip(lane, h_samples)`.

In [ ]:
def create_mask(lanes, h_samples, image_height=720, image_width=1280, line_width=5):
    mask = np.zeros((image_height, image_width), dtype=np.uint8)

    for lane in lanes:
        points = []

        for x, y in zip(lane, h_samples):
            if x == -2:
                continue
            points.append((int(x), int(y)))

        if len(points) > 1:
            cv2.polylines(mask, [np.array(points)], False, 255, line_width)

    return mask


def read_tusimple_json(json_files):
    samples = []
    for json_path in json_files:
        with open(json_path, "r") as file:
            samples.extend(json.loads(line) for line in file if line.strip())
    return samples


def load_data(dataset_path, json_files, limit=200):
    data = read_tusimple_json(json_files)
    if limit:
        data = data[:limit]

    images = []
    masks = []

    for item in tqdm(data):
        img_path = os.path.join(dataset_path, item["raw_file"])
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = create_mask(
            lanes=item["lanes"],
            h_samples=item["h_samples"],
            image_height=img.shape[0],
            image_width=img.shape[1],
        )

        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)

        images.append(img.astype(np.float32) / 255.0)
        masks.append(mask.astype(np.float32) / 255.0)

    return np.array(images), np.array(masks)[..., np.newaxis]

## 6. Загрузка данных

In [ ]:
X, Y = load_data(DATASET_PATH, JSON_FILES, limit=LIMIT)
print("Images:", X.shape)
print("Masks:", Y.shape)

## 7. U-Net модель

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x


def build_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)

    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 256)

    u1 = layers.UpSampling2D()(c3)
    u1 = layers.Concatenate()([u1, c2])
    c4 = conv_block(u1, 128)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Concatenate()([u2, c1])
    c5 = conv_block(u2, 64)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c5)

    return Model(inputs, outputs)


model = build_unet((IMG_HEIGHT, IMG_WIDTH, 3))
model.summary()

## 8. Dice и IoU метрики

In [ ]:
SMOOTH = 1e-6


def dice_coef(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    denominator = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    return (2.0 * intersection + SMOOTH) / (denominator + SMOOTH)


def iou_coef(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + SMOOTH) / (union + SMOOTH)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

## 9. Компиляция и обучение

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=bce_dice_loss,
    metrics=["accuracy", dice_coef, iou_coef],
)

history = model.fit(
    X,
    Y,
    validation_split=0.2,
    epochs=10,
    batch_size=8,
)

## 10. Графики

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(history.history["dice_coef"], label="train Dice")
plt.plot(history.history["val_dice_coef"], label="val Dice")
plt.plot(history.history["iou_coef"], label="train IoU")
plt.plot(history.history["val_iou_coef"], label="val IoU")
plt.legend()
plt.title("Segmentation metrics")

plt.show()

## 11. Проверка результата

In [ ]:
i = min(5, len(X) - 1)
pred = model.predict(X[i:i+1])[0]

plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(X[i])
plt.title("Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(Y[i].squeeze(), cmap="gray")
plt.title("True mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred.squeeze(), cmap="gray")
plt.title("Predicted mask")
plt.axis("off")

plt.show()

## 12. Сохранение модели

In [ ]:
model.save("/content/unet_tusimple.keras")